In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import MultipleLocator, FixedLocator
import seaborn as sns
import plotly.express as px
from scipy.stats import norm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import itertools


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
from sklearn.impute import SimpleImputer

import sys
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk

import json

import time

from flax import nnx
import jax.numpy as jnp
import jax
import math 
import sys
import numpy as np

In [ ]:
def downsample(iters, values, stride=5):
    return iters[::stride], values[::stride]

In [ ]:
markers = ['o', 's', 'D', '^', 'v', '<', '>', 
           'p', '*', 'h', 'H', '+', 'x', '|', 
           '_', '.', ',', '1', '2', '3', '4']

In [ ]:
colors = [
    'red', 'green', 'blue', 'black', 'magenta', 'goldenrod', 'orange', 'purple',
    'brown', 'gray', 'navy', 'teal', 'coral', 'lime', 'indigo', 'darkgreen',
    'darkblue', 'darkred', 'salmon', 'chocolate'
]

In [ ]:
linestyles = [
    '-',        # solid
    '--',       # dashed
    '-.',       # dash-dot
    ':',        # dotted

    (0, (1, 1)),             # very dense dots
    (0, (2, 1)),             # dense dashed
    (0, (3, 1, 1, 1)),       # dash-dot-dot
    (0, (5, 1)),             # medium dashes
    (0, (5, 2)),             # spaced dashes
    (0, (4, 1, 2, 1)),       # dash-dot pattern
    (0, (3, 2, 1, 2)),       # dashed + dots
    (0, (2, 2, 2, 2)),       # equal segments
    (0, (6, 3)),             # long dashes
    (0, (1, 3)),             # short spaced dashes
    (0, (3, 3, 1, 1)),       # alternating patterns
    (0, (4, 4, 1, 1)),       # longer pattern
    (0, (7, 2, 1, 2)),       # long-short
    (0, (5, 5)),             # equally spaced long dashes
    (0, (1, 2, 3, 2)),       # complex pattern
    (0, (3, 1, 1, 1, 1, 1))  # more complex dash-dot
]

In [ ]:
def info(e):
    head   = list(e.keys())[0]
    body   = list(e[head].keys())
    bias   = e[head][body[0]]
    kernel = e[head][body[1]]
    return  head, body, list(bias), list(kernel)
def real(c):
    return float(np.real(c))  
def img(c):
    return float(np.imag(c))    
def r_i(c):
    return real(c),img(c)  

def save_params(step, params, energy):
    trained_params_list.append(params.copy())
    parameters_list.append(energy.state.parameters.copy())
    iii.append(1)
    return True

In [ ]:
def plot1(e_path1, j_out1, r_out1, L, IT, l_info, l_tp, x_pos, y_pos, str_nets):
    fig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"

    # ========== Plot exact energy ==========
    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]

    # ========== Plot Jastrow data ==========
    with open(j_out1) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  

    # ========== Plot RBM data ==========
    with open(r_out1) as f:
        data = json.load(f)
    
    iters_RBM = data["Energy"]["iters"]
    mean_data = data["Energy"]["Mean"]
    
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)
    else:
        energy_RBM = mean_data

    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)

    # ========== Plotting ==========
    # Plot RBM
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=l_info[0], 
            linestyle=linestyles[0], marker=markers[0], 
            color=colors[0], markersize=5)
    
    # Plot Jastrow
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label="JASTROW", 
            linestyle=linestyles[1], marker=markers[1], 
            color=colors[1], markersize=5)

    if exact_gs_energy != 0:
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Axis configurations
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, 
                   top=True, bottom=True, left=True, right=True)
    
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    
    ax.set_ylabel("Energia")
    ax.set_xlabel("Interações")
    
    ax.text(x_pos, y_pos, f'(a) $L={L}$ {l_tp[0]}', 
            transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    # Save and show
    pathfg = f"fig/w/EGS_L_{L}_IT_{IT}_{str_nets}.png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plot4(e_path1, e_path2, 
          j_out1, r_out1, 
          j_out2, r_out2, 
          e_path3, e_path4, 
          j_out3, r_out3, 
          j_out4, r_out4, 
          L1,L3,IT, l_info, l_tp,x_pos, y_pos,
          str_nets):

    TLABEL = "FFNN"
    
    fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"
    j_out2 = j_out2 + ".log"
    r_out2 = r_out2 + ".log"
    j_out3 = j_out3 + ".log"
    r_out3 = r_out3 + ".log"
    j_out4 = j_out4 + ".log"
    r_out4 = r_out4 + ".log"
    

    # ========== Subplot (a) ==========
    ax = axs[0]


    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]


    fx = j_out1
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
   
    fx = r_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data

    
    
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[0]


    label = l_info[0]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy != 0):
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', 
                   length=2, width=0.8, top=True, bottom=True, 
                   left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    #ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylabel(r"Energia")
    0.07
    ax.text(x_pos, y_pos, '(a) $L=' + str(L1) + '$ ' + l_tp[0] , transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    # ========== Subplot (b) ==========
    ax = axs[1]
    df = pd.read_csv(e_path2)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out2
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data


        
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[1]




    label = l_info[1]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(b) $L=' + str(L1) + '$ ' + l_tp[1] , transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)




    # ========== Subplot (c) ==========
    ax = axs[2]                   
    df = pd.read_csv(e_path3)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out3
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out3
    with open(fx) as f:
        data = json.load(f)
    

    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data
    
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp =l_info[2]

    stride = 5


    label = l_info[2]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(c) $L=' + str(L3) + '$ ' + l_tp[2], transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)



    # ========== Subplot (d) ==========
    ax = axs[3]
    df = pd.read_csv(e_path4)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out4
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out4
    with open(fx) as f:
        data = json.load(f)
    
    
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data  
   
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[3]

    stride = 5

    label = l_info[3]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label='Exact')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(d) $L=' + str(L3) + '$ ' + l_tp[3], transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    
    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    pathfg = "fig/w/EGS_L_" + str(L1) + "_" + str(L3) + "_IT_" + str(IT) + "_"  + str_nets + ".png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plotmw(paths1,paths2,L):
    
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Aumentar o espaço à esquerda para acomodar a legenda externa
    fig.subplots_adjust(left=0.12, right=0.75, hspace=0.25)

    df1   = pd.read_csv(paths1[2])
    ncol1 = len(df1.columns) - 2

    df2   = pd.read_csv(paths2[2])
    ncol2 = len(df2.columns) - 2


    x1 = df1["id"]
    x2 = df2["id"]

    # ========== Subplot (a) ==========
    ax = axs[0]
    stride = 5
    i = 0
    label = r"avg (w)"
    m1 = df1.iloc[:, 1:].mean(axis=1).round(2) 
    i  = i + 1 
    x1_ds, m1_ds = downsample(x1, m1, stride)
    ax.plot(x1_ds, m1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"median (w)"
    median1 = df1.iloc[:, 1:].median(axis=1).round(2)
    i  = i + 1 
    x1_ds, median1_ds = downsample(x1, median1, stride)
    ax.plot(x1_ds, median1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"std (w)"
    std1 = df1.iloc[:, 1:].std(axis=1).round(2)
    i  = i + 1 
    x1_ds, std1_ds = downsample(x1, std1, stride)
    ax.plot(x1_ds, std1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(amp) (w)"
    range1 = (df1.iloc[:, 1:].max(axis=1) - df1.iloc[:, 1:].min(axis=1)).round(2)
    range1 = np.log10(range1)  
    i  = i + 1 
    x1_ds, range1_ds = downsample(x1, range1, stride)
    ax.plot(x1_ds, range1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(disp) (w)"
    cv1 = (df1.iloc[:, 1:].std(axis=1) / df1.iloc[:, 1:].mean(axis=1)).round(2)
    cv1 = np.log10(cv1)        # Log base 10

    i  = i + 1 
    x1_ds, cv1_ds = downsample(x1, cv1, stride)
    ax.plot(x1_ds, cv1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    all_y_values = np.concatenate([
        m1_ds,
        m1_ds + std1_ds,  # Limite superior do desvio padrão
        m1_ds - std1_ds,  # Limite inferior do desvio padrão
        median1_ds,
        range1_ds
    ])

    y_padding = 0.1 * (np.nanmax(all_y_values) - np.nanmin(all_y_values))
    y_min = np.nanmin(all_y_values) - y_padding
    y_max = np.nanmax(all_y_values) + y_padding

    ax.set_ylim(y_min, y_max)


    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_minor_locator(FixedLocator([]))
   
    
    ax.set_ylabel(r"Metrics Weights")

    
    # Mover o texto (a) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(a) $L=' + str(L) + '$ Regime Ferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom') 
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, edgecolor = 'black')
    

    # ========== Subplot (c) ==========
    x1 = x2
    ax = axs[1]
    i = 0
    label = r"avg (w)"
    m1 = df2.iloc[:, 1:].mean(axis=1).round(2) 
    i  = i + 1 
    x1_ds, m1_ds = downsample(x1, m1, stride)
    ax.plot(x1_ds, m1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"median (w)"
    median1 = df2.iloc[:, 1:].median(axis=1).round(2)
    i  = i + 1 
    x1_ds, median1_ds = downsample(x1, median1, stride)
    ax.plot(x1_ds, median1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"std (w)"
    std1 = df2.iloc[:, 1:].std(axis=1).round(2)
    i  = i + 1 
    x1_ds, std1_ds = downsample(x1, std1, stride)
    ax.plot(x1_ds, std1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(amp) (w)"
    range1 = (df2.iloc[:, 1:].max(axis=1) - df2.iloc[:, 1:].min(axis=1)).round(2)
    range1 = np.log10(range1)   
    
    i  = i + 1 
    x1_ds, range1_ds = downsample(x1, range1, stride)
    ax.plot(x1_ds, range1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(disp) (w)"
    cv1 = (df2.iloc[:, 1:].std(axis=1) / df2.iloc[:, 1:].mean(axis=1)).round(2)
    cv1 = np.log10(cv1)        # Log base 10

    i  = i + 1 
    x1_ds, cv1_ds = downsample(x1, cv1, stride)
    ax.plot(x1_ds, cv1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    # Configurações do eixo

    all_y_values = np.concatenate([
        m1_ds,
        m1_ds + std1_ds,  # Limite superior do desvio padrão
        m1_ds - std1_ds,  # Limite inferior do desvio padrão
        median1_ds,
        range1_ds
    ])

    y_padding = 0.1 * (np.nanmax(all_y_values) - np.nanmin(all_y_values))
    y_min = np.nanmin(all_y_values) - y_padding
    y_max = np.nanmax(all_y_values) + y_padding

    ax.set_ylim(y_min, y_max)
    
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.set_ylabel(r"Metrics Weights")
    ax.set_xlabel(r"Interações")

    # Mover o texto (b) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(b) $L=' + str(L) + '$ Regime Antiferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom')
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, framealpha = 1, edgecolor = 'black')
    

    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig("fig/ws/E_W_L_M_" + str(L) + "0_1.png", dpi=300, bbox_inches='tight')
    #plt.show()
    plt.close()

In [ ]:
def plotw(paths1,paths2,L, kb):
    
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Aumentar o espaço à esquerda para acomodar a legenda externa
    fig.subplots_adjust(left=0.12, right=0.75, hspace=0.25)

    df1   = pd.read_csv(paths1[kb])
    df2   = pd.read_csv(paths2[kb])

    print(paths1[kb],paths2[kb])

    #kb = 2; kernel
    if kb == 0 or kb == 1:
        ncol1 = 1
        ncol2 = 1
    else:
        ncol1 = len(df1.columns) - 2
        ncol2 = len(df2.columns) - 2
        

    # ========== Subplot (a) ==========
    ax = axs[0]
    stride = 5

    x1 = df1["id"]
    
    if kb == 0 or kb == 1:
        index = 0
        i = 0
        w1 = df1['0']
        x1_ds, w1_ds = downsample(x1, w1, stride) 
        
        label = r"bias" 
        ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    else:
        index = 0
        for i in range(0,10):
            nn = random.randint(1, ncol1) 
            w1 = df1[str(nn)]
            x1_ds, w1_ds = downsample(x1, w1, stride) 
            index = i + 1
            label = r"w" + str(index)
            ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)
   
    
    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_minor_locator(FixedLocator([]))

    if kb == 0 or kb == 1:
       ax.set_ylabel(r"Bias")
    else:
        ax.set_ylabel(r"Weights")
    
    

    
    # Mover o texto (a) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(a) $L=' + str(L) + '$ Regime Ferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom') 
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, edgecolor = 'black')
    

    # ========== Subplot (c) ==========
    ax = axs[1]


    x2 = df2["id"]

    if kb == 0 or kb == 1:
        index = 0
        i = 0
        w1 = df2['0']
        x1_ds, w1_ds = downsample(x1, w1, stride) 
        
        label = r"bias" 
        ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    else:
        
        index = 0
        for i in range(0,10):
            nn = random.randint(1, ncol2) 
            w2 = df2[str(nn)]
            x2_ds, w2_ds = downsample(x2, w2, stride) 
            index = i + 1
            label = r"w" + str(index)
            ax.plot(x2_ds, w2_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)
           
    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    if kb == 0 or kb == 1:
       ax.set_ylabel(r"Bias")
    else:
        ax.set_ylabel(r"Weights")

    ax.set_xlabel(r"Interações")

    # Mover o texto (b) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(b) $L=' + str(L) + '$ Regime Antiferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom')
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, framealpha = 1, edgecolor = 'black')
    

    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig("fig/ws/E_W_L_" + str(L) +  
                "_KB_" + str(kb) +  "_0_1.png", dpi=300, bbox_inches='tight')
    #plt.show()
    plt.close()

In [ ]:
def get_hist_max(values):
    values = values.values.astype(float)
    values = np.nan_to_num(values, nan=0.0)  # Substituir NaNs por 0
    hist, _ = np.histogram(values, bins=15, density=True)
    return hist.max()

def plot_with_normal_w(ax, data, color, title=''):
    import numpy as np
    import scipy.stats as stats

    values = data.values.astype(float)
    mean, std = values.mean(), values.std()

    # Histograma normalizado
    ax.hist(values, bins=15, color=color, alpha=0.6, density=True)

    # Curva da distribuição normal
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = stats.norm.pdf(x, mean, std)
    ax.plot(x, p, 'k', linewidth=1.5)

    # Título opcional
    if title:
        ax.set_title(title, fontsize=10)

    # Ajusta o limite y automaticamente baseado no pico da PDF
    p_max = stats.norm.pdf(mean, loc=mean, scale=std)
    ax.set_ylim(0, p_max * 1.1)  # 10% de margem
    
def plot_with_normal(ax, data, color, title, y_max):
    import scipy.stats as stats
    import numpy as np
    
    try :
        values = data.values.astype(float)
        values[~np.isfinite(values)] = 0.0
    except :
        print("erro")

     
    mean, std = values.mean(), values.std()
    
    ax.hist(values, bins=15, color=color, alpha=0.6, density=True)
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = stats.norm.pdf(x, mean, std)
    ax.plot(x, p, 'k', linewidth=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, y_max)

def plot_dist(path1, path2, ip, lp,L,it,va,y_max):
    
    df1 = pd.read_csv(path1[2])
    df2 = pd.read_csv(path2[2])

    df1.fillna(0, inplace=True)
    df2.fillna(0, inplace=True)

    lnw1 = df1.shape[1]
    lnw2 = df1.shape[1]


    i_line1 = df1.iloc[ip].iloc[1:]  
    l_line1 = df1.iloc[lp].iloc[1:]  

    i_line2 = df2.iloc[ip].iloc[1:]  
    l_line2 = df2.iloc[lp].iloc[1:]


    fig, axs = plt.subplots(1, 3, figsize=(10, 4), sharex=True)

    # Plot 1 - Distribuição Inicial
    plot_with_normal(axs[0], i_line1, 'orange', '(a) Inicial',y_max )

    axs[0].set_ylim(0, y_max)
    axs[0].set_xticks([])
    axs[0].set_xlabel('')

    axins = axs[0].inset_axes([0.1, 0.5, 0.45, 0.4])
    plot_with_normal_w(axins, i_line1, 'orange')
    axins.set_xticks([])
    axins.set_yticks([])

    # Plot 2 - Distribuição Final
    plot_with_normal(axs[1], l_line1, 'crimson', '(b) Ferromagnetismo',y_max)
    axs[1].set_ylim(0, y_max)
    axs[1].set_xticks([])
    axs[1].set_xlabel('')

    # Plot 3 - Distribuição Final - Dataset 2
    plot_with_normal(axs[2], l_line2, 'darkblue', '(b) Antiferromagnetismo',y_max)
    axs[2].set_ylim(0, y_max)
    axs[2].set_xticks([])
    axs[2].set_xlabel('')

    plt.tight_layout()

    pathimg = "distf/dif_dist_" +  str(L) +  '_' + str(it) + '_' + str(va)  
    plt.savefig(pathimg, dpi=300, bbox_inches='tight')

    
    plt.show()
    plt.close()


In [ ]:
def plot_dif(path1, path2, ip, lp):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    df1 = pd.read_csv(path1[2])
    df2 = pd.read_csv(path2[2])

    df1.fillna(0, inplace=True)
    df2.fillna(0, inplace=True)


    i_line1 = df1.iloc[ip].iloc[1:]  
    l_line1 = df1.iloc[lp].iloc[1:] 
    
    i_line2 = df2.iloc[ip].iloc[1:]  
    l_line2 = df2.iloc[lp].iloc[1:]

    dif1 = l_line1 - i_line1
    dif2 = l_line2 - i_line2

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

    for idx, (ax, dif, title) in enumerate(zip(axs, [dif1, dif2], ['Dataset 1', 'Dataset 2'])):
        bars = ax.bar(range(len(dif)), dif.values,
                      color=np.where(dif.values >= 0, 'green', 'red'),
                      alpha=0.6, edgecolor='black')

        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(f'Diferenças {title}: Final - Inicial', fontsize=12)
        ax.set_xlabel('Índice da Variável')
        ax.set_ylabel('Diferença')
        ax.grid(axis='y', alpha=0.3)

        # Define rótulos numéricos apenas em alguns ticks
        ax.set_xticks([])

    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
def plotdp(paths1, paths2, initial_point_idx, last_point_idx):
    save_path = "s4.png"

    try:
        # Data loading and validation
        df1 = pd.read_csv(paths1[2])
        if df1.empty:
            raise ValueError("Dataframe 1 is empty")

        df2 = pd.read_csv(paths2[2])
        if df2.empty:
            raise ValueError("Dataframe 2 is empty")

        # Create figure with two subplots (vertical arrangement)
        fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
        
        # Process and plot first dataset (Ferromagnético)
        plot_pca(axs[0], df1, initial_point_idx, last_point_idx, 
                title='(a) Regime Ferromagnético',
                point_labels=['Ponto Inicial', 'Ponto Final'])
        
        # Process and plot second dataset (Antiferromagnético)
        plot_pca(axs[1], df2, initial_point_idx, last_point_idx,
                title='(b) Regime Antiferromagnético',
                point_labels=['Ponto Inicial', 'Ponto Final'])

        # Configurações finais do layout
        plt.subplots_adjust(hspace=0.3, wspace=0.1)  # Aumentei o hspace para 0.3
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        plt.close()
        print(f"Figure saved to {save_path}")
        
    except Exception as e:
        print(f"Error in plot_dynamics: {str(e)}")
        raise

def plot_pca(ax, df, initial_point_idx, last_point_idx, title, point_labels):
    # Extract points
    all_points = df.iloc[:, 1:].values  # Todas as séries (sem a primeira coluna)

    # Trata valores ausentes (NaN) com média da coluna
    imputer = SimpleImputer(strategy="mean")
    all_points_imputed = imputer.fit_transform(all_points)

    # Padronização (zero média, desvio padrão 1)
    X_std = StandardScaler().fit_transform(all_points_imputed)

    # PCA
    n_components = min(2, X_std.shape[1])
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(X_std)

    # Distância Euclidiana entre o ponto inicial e final no espaço PCA
    distance = np.linalg.norm(
        principal_components[initial_point_idx] - principal_components[last_point_idx]
    )

    # Configurações do estilo do gráfico
    colors = ['green', 'red']
    mark = ['o', 's']
    line_style = 'grey'
    alpha = 0.3

    # Plot PCA Trajectory
    if n_components >= 2:
        # Full trajectory
        ax.scatter(principal_components[:, 0], principal_components[:, 1], 
                  alpha=alpha, color='blue', marker='.', label='Pontos Intermediários')
        ax.plot(principal_components[:, 0], principal_components[:, 1], 
               color=line_style, alpha=alpha, linestyle='-', label='Trajetória')
        
        # Highlight points
        ax.scatter(principal_components[initial_point_idx, 0], 
                  principal_components[initial_point_idx, 1], 
                  color=colors[0], s=100, marker=mark[0], label=point_labels[0])
        ax.scatter(principal_components[last_point_idx, 0], 
                  principal_components[last_point_idx, 1], 
                  color=colors[1], s=100, marker=mark[1], label=point_labels[1])
        
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    else:
        # 1D case
        ax.scatter(principal_components[:, 0], np.zeros(len(principal_components)), 
                  alpha=alpha, color='blue', marker='.', label='Pontos Intermediários')
        ax.plot(principal_components[:, 0], np.zeros(len(principal_components)), 
               color=line_style, alpha=alpha, linestyle='-', label='Trajetória')
        ax.scatter(principal_components[initial_point_idx, 0], 0, 
                  color=colors[0], s=100, marker=mark[0], label=point_labels[0])
        ax.scatter(principal_components[last_point_idx, 0], 0, 
                  color=colors[1], s=100, marker=mark[1], label=point_labels[1])

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    
    # Configurações dos labels
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    
    # Título e legenda
    ax.set_title(title, fontsize=12, loc='left', pad=10)
    ax.legend(fontsize=9, loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)  # Legenda fora
    ax.grid(alpha=0.3)
    
    # Ajusta o layout para acomodar a legenda
    plt.tight_layout(rect=[0, 0, 0.85, 1])  # Reduz a área útil em 15% à direita para a legenda

In [ ]:
def plot_distributions(paths1, paths2, initial_point_idx, last_point_idx, L):
    save_path = str(L) + "_distributions.png"

    try:
        # Carregar os dados
        df1 = pd.read_csv(paths1[2])
        df2 = pd.read_csv(paths2[2])

        # Criar figura com 1 linha e 2 colunas
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Plotar distribuição Ferromagnética
        plot_kde_comparison(ax1, df1, initial_point_idx, last_point_idx,
                          'Distribuição Ferromagnética')

        # Plotar distribuição Antiferromagnética
        plot_kde_comparison(ax2, df2, initial_point_idx, last_point_idx,
                          'Distribuição Antiferromagnética')

        # Ajustes finais
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Figura salva como {save_path}")

    except Exception as e:
        print(f"Erro: {str(e)}")
        raise

def plot_kde_comparison(ax, df, initial_idx, last_idx, title):
    # Extrair os pontos
    initial = df.iloc[initial_idx].iloc[1:].values
    final = df.iloc[last_idx].iloc[1:].values

    # Plotar KDE
    sns.kdeplot(initial, color='orange', label='Inicial', fill=True, alpha=0.3, ax=ax)
    sns.kdeplot(final, color='crimson', label='Final', fill=True, alpha=0.3, ax=ax)

    # Linhas de média
    ax.axvline(initial.mean(), color='orange', linestyle='--', 
              label=f'Média Inicial: {initial.mean():.2f}')
    ax.axvline(final.mean(), color='crimson', linestyle='--',
              label=f'Média Final: {final.mean():.2f}')

    # Configurações do gráfico
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel('Valores', fontsize=10)
    ax.set_ylabel('Densidade', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
    ax.tick_params(direction='in', which='both')